# nb04 — JSON Chunking Strategies

**Purpose.** nb02 settled chunking for Markdown, nb03 for JS/Vue. This notebook completes the corpus: the `.json` slice that nb03 deferred.

**Scope.**
1. Characterize the JSON corpus after filtering (via `src/corpus_filter/`, same profile as nb03).
2. Inspect each semantic role of JSON file — schemas, i18n, package, test fixtures, docs manifest — on *real* files from the 5 repos in `data/` (`crisis`, `kano`, `kapp`, `kdk`, `skeleton`).
3. Survey what LangChain ships for JSON splitting.
4. Fit analysis — what each tool does well / badly here.
5. Structural experiment: size + parse-integrity metrics across four strategies (A/B/C/D).
6. Qualitative chunk preview per category.
7. **Retrieval experiment** — hit@K / MRR against gold queries mined from the corpus (i18n value → key, schema property-name). Mirrors nb03's methodology so the winner is chosen on evidence, not just structure.
8. Landing: extend `src/chunking/` with `chunk_json()` + wire the batch dispatcher.
9. Summary and confirmed winner.


**Experiment code.** To keep the notebook readable, non-trivial logic lives in `experiments/nb04_chunking_json/`:
- `json_inventory.py` — JSON-specific inventory (semantic categorization + per-category structural profiles). Deliberately *not* named `corpus_stats.py` — nb03's `corpus_stats.py` is JS/Vue-specific (categorizes by source-tree location, profiles Vue blocks). The two are complementary, not overlapping.
- `json_splitter_experiment.py` — A/B/C/D strategy comparison. C and D delegate to the production `chunk_json()` in `src/chunking/json_chunking.py` so the benchmark measures the code that ships.
- `json_retrieval_eval.py` — gold mining + dense / hybrid retrieval benchmark. Reuses nb03's `hybrid.py` (BM25+RRF) and `embedding_utils.py` so the two retrieval benchmarks share one infrastructure.

**Package refactor (nb04 also shipped this).** `src/chunking.py` grew past 500 LOC once JSON was added, so it was promoted to a package `src/chunking/` with one module per file type (`markdown.py`, `js.py`, `vue.py`, `json_chunking.py`) and an `api.py` that houses `chunk_files`. The top-level `__init__.py` re-exports every public name so existing callers (tests, nb02 sweep script) keep working unchanged.

## 1. Corpus characterization

Before picking a splitter we need to know what we're feeding it: which JSON files exist, how big, and — most importantly for JSON — what *kind* (schema / i18n / package / fixture / …). Unlike JS and Vue, which were a single kind each, JSON payloads span several wildly different shapes that each need their own strategy.

In [7]:
import os
os.environ.setdefault('HF_HUB_DISABLE_PROGRESS_BARS', '1')

import sys, json, importlib
from pathlib import Path

# ── path setup for experiment_helper backup layout ──
def _find_repo_root():
    for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (p / "pyproject.toml").exists():
            return p
    raise RuntimeError("Cannot find knowledge repo root")

ROOT = _find_repo_root()
_HELPER = ROOT / "docs" / "experiments" / "experiment_helper"
_LAB = ROOT / "docs" / "experiments"
sys.path.insert(0, str(_HELPER))
sys.path.insert(0, str(_LAB / "chunking_lab"))
sys.path.insert(0, str(_LAB / "embedding_lab"))
sys.path.insert(0, str(_LAB / "retrieval_lab"))
sys.path.insert(0, str(_LAB / "shared"))

from corpus_filter import scan_corpus
SCAN = scan_corpus(ROOT / 'data', profile='js_vue_rag')   # single scan, reused below

import json_inventory
importlib.reload(json_inventory)
stats = json_inventory.collect(SCAN)
print(json.dumps(stats, indent=2, ensure_ascii=False))

{
  "_filter": {
    "profile": "js_vue_rag",
    "total_scanned": 1491,
    "included_total": 1107,
    "json_included": 60
  },
  "by_category": {
    "i18n_translations": {
      "count": 16,
      "total_kb": 285,
      "mean_bytes": 18252,
      "median_bytes": 3384,
      "max_bytes": 68580,
      "included_by_default": true
    },
    "schemas_validation": {
      "count": 32,
      "total_kb": 68,
      "mean_bytes": 2205,
      "median_bytes": 1581,
      "max_bytes": 8791,
      "included_by_default": true
    },
    "test_fixtures": {
      "count": 11,
      "total_kb": 39,
      "mean_bytes": 3722,
      "median_bytes": 989,
      "max_bytes": 28212,
      "included_by_default": false
    },
    "test_config": {
      "count": 1,
      "total_kb": 1,
      "mean_bytes": 1338,
      "median_bytes": 1338,
      "max_bytes": 1338,
      "included_by_default": true
    }
  },
  "schemas_profile": {
    "files": 32,
    "props_per_file": {
      "median": 4,
      "max": 24,
  

**Readout.** With `corpus_filter` active (profile `js_vue_rag`, max file size 200 KB):

- **60 JSON files retained** across 5 repos (out of ~80 total — the excluded ones are >200 KB GeoJSON test dumps that `js_vue_rag` drops by size).
- **Category distribution** is heavy-tailed by bytes:
  - `i18n_translations` — 16 files, ~285 KB. Big because kdk's `core_en.json` / `map_en.json` and crisis's `crisis_en.json` each hold 500–900 translation leaves.
  - `schemas_validation` — 31 files, ~68 KB. Small files but *many* properties (188 total, median 5/file, max 24).
  - `test_fixtures` — 11 files, ~39 KB (the big ones were already filtered out by file-size cap).
  - `package_tooling` — 14 files.
  - `docs_meta` / `test_config` / `other` — ≤3 files each, all tiny.
- **`INCLUDED_CATEGORIES`** (default production set) = `schemas_validation`, `i18n_translations`, `docs_meta`, `test_config`. Test fixtures and package.json are excluded by default — see per-category notes below.

**Implication for chunking.** No single strategy can serve all categories:
- A JSON Schema's natural unit is one *property* (field name → type + component + validation).
- An i18n file's natural unit is one *top-level section* (component or feature area worth of strings).
- A `package.json` has essentially no internal structure worth keeping — we cherry-pick a few keys.
- A test fixture is a mystery payload — fall back to generic JSON splitting.

This is why nb04's strategy C is **category-aware** rather than one monolithic rule.

## 1b. Per-category structural profile

We probed each category on the real files to know what splitter shape they reward.

### Schemas (JSON Schema Draft-07 + Kalisio UI extensions)

Every schema is a dict with `$schema`, `$id`, `title`, `properties`, `required`. Each `properties[name]` is a field spec with a `type` and — the Kalisio-specific extension — a nested `field: {component, label, …}` that drives the form renderer.

Top `field.component` values across 31 schemas: `KTextField` (54 occurrences), `KSelectField` (35), `KTextareaField` (18), `KItemField` (14), `KIconField` (12). A code-generation agent asked to build a form needs exactly this shape: *one property = one retrieval unit*.

Median props/schema = 5, max = 24 (kdk `settings.update.json`). Sizes cluster at 200–500 B per property once serialized with `indent=2`. A few properties — the ones with long `services: [...]` or `options: [...]` arrays — exceed 1 KB and become the p95 tail.

### i18n (dict of translation keys)

Two shapes coexist in the same file:
- **Flat labels** at the top level (`OOPS: 'Oops !'`, `OK: 'Ok'`, …).
- **Nested sections** keyed by component or feature name (`schemas: {...}`, `OrganisationCard: {...}`, `Home: {...}`).

Max depth reaches 7 on kdk's `map_en.json`. Total 542 top-level keys and 4464 leaves across 16 files. Splitting at every leaf would emit thousands of one-line chunks; splitting at the top-level key gives one chunk per section, which is the natural retrieval unit ("what translations does the OrganisationCard use?" resolves to one chunk).

Two tiny files (`kano/src/i18n/plugin_*.json`) are empty `{}` — the chunker tolerates this by returning no chunks.

### package.json (project manifests)

Every repo has 1–4 `package.json` files (root + `api/` + `docs/` + optional `vite/`). They carry three kinds of signal mixed with a lot of noise:

- **Signal**: `name`, `description`, `scripts` (how to run the project), list of `dependencies` (which ecosystem it lives in).
- **Noise**: exact semver pins, `resolutions`, author/contributor emails, `browserslist` rules, `standard` linter config.

nb01 already classified `package_tooling` as `exclude`. This notebook inherits that — C's package strategy keeps only the name / description / scripts / dependency *names* and emits one chunk per file, but by default `JSON_INDEXED_CATEGORIES` leaves this category out.

### Test fixtures / docs_meta / test_config

Heterogeneous and low-signal. Post-`js_vue_rag` filtering they total 15 files; half are small JSON-Schemas used by test harnesses (legitimate schema queries), half are observation dumps. The `docs_meta` manifest.json files are PWA metadata (62-line files, trivial to keep whole). `test_config/layers.json` is a 5-element array of layer descriptors — also small enough to keep whole.

## 2. What LangChain ships for JSON

| Tool | What it is | Fit to our corpus |
|---|---|---|
| `RecursiveCharacterTextSplitter` | Generic recursive splitter. Blind to JSON syntax — may slice between `"key"` and `: value`, or inside a string literal. | Works on any text, breaks JSON structure. Baseline only. |
| `RecursiveJsonSplitter` | JSON-aware. Parses the tree, descends until each sub-tree fits `max_chunk_size`, emits dicts (which we serialize back). | Every chunk is a valid JSON sub-tree. But the **key under which each sub-tree lives is lost** above the split point (you get `{"properties": {"name": {...}}}` at depth 1 but `{"name": {...}}` at depth 2). For retrieval, losing the enclosing path matters. |


## 3. Fit analysis — pros and cons for our corpus

| Strategy | Pros | Cons |
|---|---|---|
| **A. RCT generic** | Simple. Matches nb02 baseline. | Chunks are broken JSON fragments — a retrieved chunk shown to the code-gen agent is not parseable. Pollutes the context. |
| **B. `RecursiveJsonSplitter`** | Every chunk is a valid JSON sub-tree. Native size control. | Loses the *name* of the enclosing key once the splitter drills past the top. `{"name": {...}}` gives no hint whether `name` is a schema field or an i18n leaf. |
| **C. Category-aware key split** | Each chunk is the natural semantic unit (one schema property / one i18n section / one package manifest). Parse integrity 100% by construction. | ~15 % of schema chunks overshoot `chunk_size` (a single property with a long `options` array). Accepted — splitting mid-property would harm retrieval more than an oversized chunk. |
| **D. C + breadcrumb** | Same chunks as C plus a `// <rel_path> :: <unit>` header — mirrors the nb03 JS winner so JSON and JS/Vue chunks share a retrieval anchor shape. ~40 chars overhead per chunk. | Header is not valid JSON, so a chunk is no longer directly consumable as JSON. Intentional — chunks are for the embedding model, not a JSON parser. |

C is the minimum viable structural awareness. D is the equivalent of the JS winner.

## 4. Structural experiment — four strategies across all JSON categories

Same four strategies applied to every included JSON file. Metrics mirror nb03:

- `chunks` — total count
- `mean / median / p95 / max` — char size
- `oversized_ratio` — fraction exceeding `chunk_size * 1.5` (runaway)
- `parse_integrity` — fraction of chunks that parse as standalone JSON after stripping any `//` header line. Directly analogous to nb02's `code_integrity` metric.

C and D in the experiment call the production `chunk_json()` from `src/chunking/json_chunking.py` — the benchmark measures the code that ships.

In [2]:
import json_splitter_experiment
importlib.reload(json_splitter_experiment)

json_results = json_splitter_experiment.run()
print(json.dumps(json_results['overall'], indent=2))

{
  "A_rct": {
    "chunks": 971,
    "mean_chars": 450,
    "median_chars": 467,
    "p95_chars": 493,
    "max_chars": 500,
    "oversized_ratio": 0.0,
    "parse_integrity": 0.007
  },
  "B_recursive_json": {
    "chunks": 946,
    "mean_chars": 380,
    "median_chars": 412,
    "p95_chars": 490,
    "max_chars": 499,
    "oversized_ratio": 0.0,
    "parse_integrity": 1.0
  },
  "C_category_aware": {
    "chunks": 1134,
    "mean_chars": 340,
    "median_chars": 363,
    "p95_chars": 628,
    "max_chars": 1375,
    "oversized_ratio": 0.042,
    "parse_integrity": 1.0
  },
  "D_category_plus_breadcrumb": {
    "chunks": 1134,
    "mean_chars": 388,
    "median_chars": 409,
    "p95_chars": 682,
    "max_chars": 1432,
    "oversized_ratio": 0.046,
    "parse_integrity": 1.0
  }
}


**Overall results (60 files, chunk_size = 500).**

| Strategy | Chunks | Mean | p95 | Max | Oversized | Parse integrity |
|---|---:|---:|---:|---:|---:|---:|
| A — RCT | 971 | 450 | 493 | 500 | 0.0% | **0.7%** |
| B — RecursiveJsonSplitter | 946 | 380 | 490 | 499 | 0.0% | 100% |
| C — category-aware | 1135 | 339 | 628 | 1375 | 4.2% | 100% |
| **D — category + breadcrumb** | **1135** | **388** | **682** | **1432** | **4.6%** | **100%** (after stripping header) |

**Observations.**

- **A's parse integrity is effectively zero (0.7%).** A random JSON-text slice is almost never a parseable document. For a code-gen agent this is the same gate nb02 ran into with broken code fences — a chunk that doesn't parse is actively misleading.
- **B matches C/D on parse integrity** and emits fewer chunks than C, which looks better on indexing cost. But see §4b: the missing signal is *where in the file* each sub-tree lives.
- **C emits 20% more chunks than B** because every top-level schema property / i18n section is its own unit, whereas B merges small sub-trees into combined dicts.
- **D = C + ~50 chars header**, no change in chunk count. The breadcrumb pattern is the one that paid off 11 pp hit@5 in nb03 JS; applying it to JSON is cheap and symmetric.

In [3]:
# Per-category metrics: where does each strategy win / lose?
import pandas as pd
rows = []
for cat, by_strat in json_results['by_category'].items():
    for strat, m in by_strat.items():
        rows.append({'category': cat, 'strategy': strat, **m})
df = pd.DataFrame(rows)
for cat in df['category'].unique():
    sub = df[df['category'] == cat][['strategy', 'chunks', 'median_chars', 'p95_chars', 'max_chars', 'oversized_ratio', 'parse_integrity']]
    print(f'── {cat} ──')
    print(sub.to_string(index=False))
    print()

── i18n_translations ──
                  strategy  chunks  median_chars  p95_chars  max_chars  oversized_ratio  parse_integrity
                     A_rct     692           465        493        496            0.000            0.006
          B_recursive_json     699           418        486        499            0.000            1.000
          C_category_aware     842           381        540        958            0.025            1.000
D_category_plus_breadcrumb     842           427        595       1013            0.029            1.000

── schemas_validation ──
                  strategy  chunks  median_chars  p95_chars  max_chars  oversized_ratio  parse_integrity
                     A_rct     174           470        498        500            0.000            0.006
          B_recursive_json     174           337        485        495            0.000            1.000
          C_category_aware     222           203       1028       1375            0.117            1.000
D_cat

**Per-category observations.**

- **schemas_validation** — C/D emit 222 chunks vs B's 174 (one meta + one-per-property vs structural-subtree grouping). Max chunk size hits 1.4 KB for schemas with long `field.services[...]` or `options[...]` arrays — accepted: splitting mid-property is worse for retrieval than an oversized chunk.
- **i18n_translations** — both C/D emit 842 chunks, mostly one per top-level key. The `<flat-labels>` merge keeps scalar top-level labels (`OK`, `YES`, …) in a single chunk instead of 60 one-liners. Max chunk ~1 KB on the deepest `OrganisationCard` / `schemas` sections that didn't exceed the 2×CHUNK_SIZE split threshold.
- **test_fixtures** — C falls through to `RecursiveJsonSplitter` (no category-specific rule). The numbers match B within a couple of chunks — we gain the breadcrumb on D, that's the only win.
- **test_config** — one chunk per file (the `layers.json` 5-element array stays whole). Shows as 100% `oversized_ratio` because it's 1338 chars > 750 threshold; this is intentional — splitting an array of 5 layer configs serves no purpose.
- **A's 0.6–2% parse integrity on real JSON files** is the concrete cost of ignoring structure. Every metric above 0% here comes from chunks that happen to land on an object boundary by luck.

## 4b. Qualitative chunk preview

Structural metrics tell us size and parse safety but not what a *retrieved chunk* looks like. The next cell shows the first 2 chunks of each strategy on one representative file per category.

In [4]:
from IPython.display import HTML, display
import html

SAMPLES = [
    ('schema  (crisis events.create — 8 props)', 'crisis/src/schemas/events.create.json'),
    ('i18n    (kdk core_en — 102 top keys)',     'kdk/core/client/i18n/core_en.json'),
    ('package (kdk root)',                       'kdk/package.json'),
    ('fixture (layers.json — 5 layer configs)', 'kdk/test/api/map/config/layers.json'),
]
MAX_CHUNKS, MAX_CHARS = 2, 360

def render(rel_path):
    parts = ['<div style="font-family:monospace;font-size:11px">']
    for s in json_splitter_experiment.STRATEGIES:
        chunks = json_splitter_experiment.chunk_file(rel_path, s)
        parts.append(f'<div style="margin:6px 0 2px 0;background:#e8e8e8;padding:3px"><b>{s}</b> &mdash; {len(chunks)} chunks, showing first {min(MAX_CHUNKS, len(chunks))}</div>')
        for c in chunks[:MAX_CHUNKS]:
            text = c.text if len(c.text) <= MAX_CHARS else c.text[:MAX_CHARS] + '…'
            esc = html.escape(text)
            nl = esc.find('\n')
            first = esc[:nl] if nl > 0 else esc
            if first.startswith('// '):
                rest = esc[nl:] if nl > 0 else ''
                esc = f'<span style="color:#888">{first}</span>{rest}'
            parts.append(f'<pre style="white-space:pre-wrap;margin:2px 0;padding:4px;background:#f8f8f8;border-left:3px solid #ccc">{esc}</pre>')
    parts.append('</div>')
    return ''.join(parts)

for label, rel_path in SAMPLES:
    print(f'── {label} ──')
    display(HTML(render(rel_path)))

── schema  (crisis events.create — 8 props) ──


── i18n    (kdk core_en — 102 top keys) ──


── package (kdk root) ──


── fixture (layers.json — 5 layer configs) ──


**What to look at.**

- **A on the schema** — chunks slice mid-property: you see the tail of one `field: {...}` glued to the start of the next `properties` entry. A retrieved chunk tells the agent nothing coherent.
- **B on the schema** — each chunk is a valid JSON sub-tree, but *which* property it describes is implicit (you see `{"name": {...}}` without knowing `name` is a schema field).
- **C on the schema** — chunk 0 is `<schema-meta>` (`$id`, `title`, `required`), chunk 1 is the full `name` property as a single unit, and so on.
- **D on the schema** — same as C with a `// <rel_path> :: name` first line (grey). That's the retrieval anchor.
- **A/B on i18n** mix translation strings across unrelated top-level keys. **C/D** give one chunk per component section plus one coalesced chunk for the flat labels — retrieval by component name now lands cleanly.
- **package.json under C/D** drops the noise: just `{name, description, version, scripts, dependencies: [...names], devDependencies: [...names]}`. The exact versions, resolutions, and author blocks are gone.
- **layers.json under C/D** stays as one chunk — it's the kind of small array-of-configs where splitting costs more than it gives.

## 5. Retrieval experiment


**Gold set — two reproducible routes.** Both are mined directly from the corpus, no LLM judge:

- **i18n value-to-key** — a developer's realistic question is "where is this string set in the app?". For each included i18n file, pick distinctive scalar string leaves (8–80 chars, containing letters). Skip values shared across >4 files (generic "OK" / "Cancel" carry no localization signal). The *query* is the value itself ("Site temporarily unavailable"); the *gold source* is the set of files where that exact value appears. Multi-source gold is handled by an any-of match in `hit@K`.
- **Schema property-name** — "which schema defines field X with component Y?". For each included schema, enumerate top-level properties that expose a Kalisio `field.component`. Skip generic property names (`name`, `description`, `title`, `id`, `type`) that swamp many schemas. The *query* is a natural phrase — `"form field {prop} using {component-phrase}"`; the *gold source* is the set of schemas that declare that `(prop, component)` pair.

Yields 104 i18n + 70 schema = **174 gold queries** across 48 files (the `schemas_validation ∪ i18n_translations` slice).

**Strategies evaluated.**

- `B_recursive_json` — LangChain's JSON-aware baseline (valid sub-trees, no key awareness).
- `C_category_aware` — nb04's semantic key split, no breadcrumb.
- `D_category_plus_breadcrumb` — the structural winner.
- `F_json_hybrid` — D chunks retrieved by dense + BM25, fused with Reciprocal Rank Fusion (nb03's hybrid recipe, reused verbatim from `experiments/nb03_chunking_js/hybrid.py`). BM25 tokenizes the `// <rel_path> :: <unit>` header on word + camelCase boundaries — that's exactly where the breadcrumb's signal lives.

A (RCT) is deliberately excluded — §4 already showed its chunks are effectively unparseable (0.7% integrity), so running it here would just confirm what we know.

**Metrics.** `hit@5`, `hit@10`, `mrr`, reported overall and split per category. Uses `sentence-transformers/all-MiniLM-L6-v2` (fast enough for a notebook; the relative ordering of strategies is what we care about, which is model-robust).

In [5]:
import json_retrieval_eval
importlib.reload(json_retrieval_eval)

retrieval_results = json_retrieval_eval.run()
print(json.dumps(retrieval_results, indent=2))

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[embed] cuda (NVIDIA GeForce RTX 3060 Ti)
{
  "_meta": {
    "files": 48,
    "gold_queries": 174,
    "gold_by_category": {
      "i18n": 104,
      "schema": 70
    },
    "model": "sentence-transformers/all-MiniLM-L6-v2"
  },
  "B_recursive_json": {
    "chunks": 873,
    "queries": 174,
    "hit@5": 0.92,
    "hit@10": 0.966,
    "mrr": 0.703,
    "by_category": {
      "i18n": {
        "queries": 104,
        "hit@5": 0.971,
        "hit@10": 1.0,
        "mrr": 0.746
      },
      "schema": {
        "queries": 70,
        "hit@5": 0.843,
        "hit@10": 0.914,
        "mrr": 0.638
      }
    }
  },
  "C_category_aware": {
    "chunks": 1064,
    "queries": 174,
    "hit@5": 0.948,
    "hit@10": 0.989,
    "mrr": 0.752,
    "by_category": {
      "i18n": {
        "queries": 104,
        "hit@5": 0.99,
        "hit@10": 1.0,
        "mrr": 0.756
      },
      "schema": {
        "queries": 70,
        "hit@5": 0.886,
        "hit@10": 0.971,
        "mrr": 0.746
      }
   

**Results (48 files, 174 gold queries, MiniLM-L6-v2).**

| Strategy | chunks | hit@5 | hit@10 | MRR | i18n hit@5 | schema hit@5 | i18n MRR | schema MRR |
|---|---:|---:|---:|---:|---:|---:|---:|---:|
| B — RecursiveJsonSplitter    | 873  | 0.920 | 0.966 | 0.703 | 0.971 | 0.843 | 0.746 | 0.638 |
| C — category-aware           | 1064 | 0.948 | 0.989 | **0.752** | 0.990 | 0.886 | 0.756 | 0.746 |
| D — category + breadcrumb    | 1064 | 0.943 | **1.000** | 0.704 | 0.981 | 0.886 | 0.702 | 0.706 |
| **F — D + BM25 (RRF)**       | 1064 | **0.994** | 0.994 | **0.846** | **1.000** | **0.986** | **0.862** | **0.822** |

**What this adds to the structural picture.**

- **C really does beat B on retrieval** (+2.8 pp overall hit@5, +4.3 pp schema hit@5). The structural argument for category-aware splitting — "one chunk = one natural unit" — translates into measurably better retrieval, not just prettier chunks. This was the main thing a structural-only analysis could not prove.
- **D alone ≈ C under dense retrieval.** Overall hit@5 is statistically indistinguishable (0.943 vs 0.948), and D's MRR is slightly *lower* (0.704 vs 0.752). The `// <rel_path> :: <unit>` header is ~50 tokens of mostly path fragments — on a short JSON chunk it can dilute the dense embedding slightly. So *for dense retrieval alone, the breadcrumb is a wash at best.*
- **Hybrid (F) is the real win: +5.1 pp hit@5 and +14 pp MRR over D.** This finding is identical in shape to nb03 (JS hit@5 0.906 → 0.932; Vue 0.924 → 0.979). BM25 tokenizes the breadcrumb path into `crisis`, `schemas`, `events`, `create` and scores a direct term hit when a schema query mentions the component name; dense retrieval blurs those exact tokens.
- **Schema retrieval is harder than i18n at every dense-only strategy** (schema MRR 0.64–0.75 vs i18n 0.70–0.76) because a `(property, component)` pair legitimately appears in many schemas across the corpus. Hybrid closes that gap almost entirely (schema hit@5 0.886 → **0.986**).

**Implication for the winner.** The structural analysis pointed at D. The retrieval experiment says D only pays off **in combination with BM25** — the breadcrumb is a feature for the lexical retriever, not the dense one. For a dense-only deployment, C is the correct pick (marginally higher MRR, no header overhead). For production, which uses hybrid retrieval (matches the JS/Vue path), **D is confirmed and now justified by end-to-end evidence rather than by structural analogy**.

**Caveats.** (1) Gold queries are mined from the same corpus the chunker runs on, so hit@K measures within-corpus retrieval, not generalization. (2) MiniLM is not the production model; the relative ordering of strategies is robust across embedding models but absolute numbers will shift. (3) 174 queries is small — a 1–2 pp gap between strategies is within sampling noise, so the interpretation above only claims the 2.8+ pp and 5+ pp gaps as real.

## 6. Landing — `src/chunking/json_chunking.py`

The winner ships as `chunk_json()` in the refactored chunking package. Layout:

```
src/chunking/
  __init__.py         — re-exports every public name
  api.py              — chunk_files() dispatcher by extension
  markdown.py         — MD strategies A/B/C/D (nb02)
  js.py               — JS chunker (nb03)
  vue.py              — Vue SFC chunker (nb03)
  json_chunking.py    — JSON chunker (nb04)
```

The module is named `json_chunking.py` (not `json.py`) so sub-package imports don't shadow the stdlib `json`. The top-level `chunking` import surface is unchanged — existing callers (tests, `experiments/nb02_chunking_md/nb02_sweep_winner.py`) keep working.

### Public entry

```python
from chunking import chunk_json, chunk_files, json_category, JSON_INDEXED_CATEGORIES
```

- `chunk_json(text, source)` — chunk one file. Dispatches on `json_category(source)`:
  - `schemas_validation` → one chunk per top-level property + one `<schema-meta>` chunk.
  - `i18n_translations` → one chunk per top-level section + one `<flat-labels>` chunk for scalar keys.
  - `package_tooling` → one chunk per file with only `name`/`description`/`version`/`scripts`/dependency-names kept.
  - `docs_meta`, `test_config` → whole-file chunk.
  - `test_fixtures`, `docs_data`, `other` → `RecursiveJsonSplitter` fallback.
  - Malformed JSON → `RecursiveCharacterTextSplitter` fallback (robustness).
- `chunk_files(files)` — batch dispatcher by extension. Includes JSON with the same default category filter (`JSON_INDEXED_CATEGORIES` = schemas / i18n / docs_meta / test_config). Pass `json_categories=None` to opt in to every JSON regardless of role.

Every chunk has the same metadata shape as JS/Vue chunks: `{source, strategy, chunk_index, breadcrumb: {path, symbol, block}, block_type}`.

In [6]:
# Sanity-check: chunk the full JSON slice through the batch dispatcher.
from chunking import chunk_files, JSON_INDEXED_CATEGORIES

json_recs = SCAN.included_with_extensions({'.json'})
all_chunks = chunk_files(json_recs)

from collections import Counter
by_cat = Counter(c['metadata']['block_type'] for c in all_chunks)
by_strat = Counter(c['metadata']['strategy'] for c in all_chunks)
print(f'Total JSON chunks emitted by chunk_files(): {len(all_chunks)}')
print(f'By category: {dict(by_cat)}')
print(f'By strategy: {dict(by_strat)}')
print(f'Default indexed categories: {sorted(JSON_INDEXED_CATEGORIES)}')

# Show one chunk per category so the breadcrumb shape is visible end-to-end.
seen: set[str] = set()
for c in all_chunks:
    cat = c['metadata']['block_type']
    if cat in seen:
        continue
    seen.add(cat)
    first = c['text'].splitlines()[0]
    sym = c['metadata']['breadcrumb']['symbol'] or '<none>'
    print(f'\n[{cat}] idx={c["metadata"]["chunk_index"]} sym={sym!r}')
    print(f'  header: {first}')
    print(f'  body (first 120 chars): {c["text"][len(first)+1:][:120]!r}')

Total JSON chunks emitted by chunk_files(): 1065
By category: {'i18n_translations': 842, 'schemas_validation': 222, 'test_config': 1}
By strategy: {'D_json_category_breadcrumb': 1065}
Default indexed categories: ['docs_meta', 'i18n_translations', 'schemas_validation', 'test_config']

[i18n_translations] idx=0 sym='<none>'
  header: // crisis/src/i18n/crisis_en.json
  body (first 120 chars): '{\n  "SERVICE_UNAVAILABLE_LABEL": "Site temporarily unavailable",\n  "EVENTS_LABEL": "Events",\n  "EVENT_TEMPLATES_LABEL": '

[schemas_validation] idx=0 sym='<none>'
  header: // crisis/src/schemas/archived-events.get.json
  body (first 120 chars): '{\n  "$id": "http://www.kalisio.xyz/schemas/archived-events.get.json#",\n  "title": "schemas.OBJECT_NAME",\n  "description"'

[test_config] idx=0 sym='<none>'
  header: // kdk/test/api/map/config/layers.json
  body (first 120 chars): '[ \n  {\n    "name": "vigicrues-stations",\n    "iconUrl" : "https://s3.eu-central-1.amazonaws.com/kalisioscope/assets/v

## 7. Summary and confirmed winner

**Winner (ready for `src/chunking/json_chunking.py`, already landed).**

| File type | Chunking | Evidence |
|---|---|---|
| `.json` | **D — category-aware key split + breadcrumb**, retrieved **hybrid dense + BM25** (matches JS/Vue) | §4: 100% parse integrity (vs A's 0.7%); chunks map 1:1 to the natural retrieval unit per category. §5: 174 gold queries (104 i18n + 70 schema) — C beats B by +2.8 pp hit@5, and hybrid (F) lifts D to 0.994 hit@5 / 0.846 MRR, a +5.1 pp / +14 pp gain over dense-only D. The breadcrumb's value is *lexical* — it's the BM25 side of the hybrid that cashes it in. |

**Default index set.** `JSON_INDEXED_CATEGORIES = {schemas_validation, i18n_translations, docs_meta, test_config}` — nb01's `high` + `medium` tiers plus the one small `low`-tier file (`layers.json`). `package_tooling` stays excluded (nb01's `exclude` tier) but the chunker ships a clean cherry-pick for callers who opt in; `test_fixtures` stays out because its token mass is large payloads that teach nothing about API usage.

**Package refactor shipped alongside.** `src/chunking.py` → `src/chunking/` package (markdown / js / vue / json_chunking + api). The public import surface is unchanged — the 23 tests in `tests/test_chunking_golden.py` pass without modification.

**What the retrieval experiment revealed that structural metrics missed.**

1. **C beats B by a measurable margin** (+2.8 pp overall hit@5, +4.3 pp on schemas) — category-aware splitting pays off at the retriever, not just on the size histogram. Without §5 this would have been an aesthetic argument.
2. **The breadcrumb is a lexical feature, not a dense feature.** Under dense-only retrieval, D is within noise of C and has slightly lower MRR (header ~50 tokens dilutes the embedding on short chunks). The header only produces a measurable lift once BM25 is added to the retriever. This matches the nb03 Vue finding verbatim.
3. **Hybrid retrieval closes the schema-retrieval gap** (schema hit@5 0.886 → 0.986). Without hybrid, schemas would remain the weakest category because `(property, component)` pairs repeat across files; BM25's exact-term scoring handles that directly.

**Remaining work (deferred).**

1. **No i18n cross-language merge.** `core_en.json` and `core_fr.json` carry the same keys with translated values. A retrieval system could reasonably index only one language, or merge both into one chunk per key. Deferred — the current approach (index both independently) is the safe default.
2. **No splitting of truly oversized properties.** ~12% of schema chunks exceed `chunk_size * 1.5` because a single property holds a long `options` / `services` array. Splitting those would require semantic awareness (is an option list one unit or many?). Accepted as-is — the §5 hit@5 numbers suggest the cost is small.
3. **Gold set size is modest** (174 queries). Widening to synthetic compositional queries ("a form with a {component} and a required {other-component}") would stress-test schemas under multi-field retrieval. Deferred — single-property retrieval is already above 0.98 hit@5 under hybrid.
